# How to use Sugar

## Initial Setup

### Uninstall cuda version if current is not cuda toolkit 11.8

```sh
sudo apt-get --purge remove "*cublas*" "cuda*" "nsight*" 
sudo apt-get --purge remove "*nvidia*"
sudo rm -rf /usr/local/cuda*
sudo apt-get purge nvidia*
sudo apt-get autoremove
sudo apt-get autoclean
sudo rm -rf /usr/local/cuda*
sudo apt-get --purge remove "*cublas*" "*cufft*" "*curand*" \
 "*cusolver*" "*cusparse*" "*npp*" "*nvjpeg*" "cuda*" "nsight*" 
sudo apt-get --purge remove "*nvidia*"
sudo apt-get autoremove
# Below nvidia-* and libnvidia-* removes drivers also. Better to remove everything and reinstall. The libcudnn8* removed cuDNN.
sudo apt-get --purge remove cuda-* nvidia-* gds-tools-* libcublas-* libcufft-* libcufile-* libcurand-* libcusolver-* libcusparse-* libnpp-* libnvidia-* libnvjitlink-* libnvjpeg-* nsight* nvidia-* libnvidia-* libcudnn8*

# Also run below which gets rid of CUDA 10 and prior stuff.
sudo apt-get --purge remove "*cublas*" "*cufft*" "*curand*" "*cusolver*" "*cusparse*" "*npp*" "*nvjpeg*" "cuda*" "nsight*"

# Cleanup uninstall
sudo apt-get autoremove
sudo apt-get autoclean

# remove cuda directories
sudo rm -rf /usr/local/cuda*

# remove from dpkg
sudo dpkg -r cuda
sudo dpkg -r $(dpkg -l | grep '^ii  cudnn' | awk '{print $2}')

### If you have previous installation remove it first. 
sudo apt-get purge nvidia*
sudo apt remove nvidia-*
sudo rm /etc/apt/sources.list.d/cuda*
sudo apt-get autoremove && sudo apt-get autoclean
sudo rm -rf /usr/local/cuda*

# system update
sudo apt-get update
sudo apt-get upgrade
```

### Install cuda 11.8 - part 1

```sh
sudo apt update
sudo apt upgrade
sudo apt install ubuntu-drivers-common
sudo ubuntu-drivers devices
sudo apt install nvidia-driver-550
sudo reboot now
```

### Install cuda 11.8 - part 2

```sh
nvidia-smi
wget https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-ubuntu2204.pin
sudo mv cuda-ubuntu2204.pin /etc/apt/preferences.d/cuda-repository-pin-600
wget https://developer.download.nvidia.com/compute/cuda/11.7.1/local_installers/cuda-repo-ubuntu2204-11-7-local_11.7.1-515.65.01-1_amd64.deb
sudo dpkg -i cuda-repo-ubuntu2204-11-7-local_11.7.1-515.65.01-1_amd64.deb
sudo cp /var/cuda-repo-ubuntu2204-11-7-local/cuda-*-keyring.gpg /usr/share/keyrings/
sudo apt-get update
sudo apt install cuda-11-8
sudo reboot now
```

In the file ~/.bashrc add or change it in case of already existing:
```sh
# CUDA PATH
export PATH=/usr/local/cuda/bin${PATH:+:${PATH}}
export LD_LIBRARY_PATH=/usr/local/cuda-11.8/lib64\
                         ${LD_LIBRARY_PATH:+:${LD_LIBRARY_PATH}}
```

Reload the file in command line:
```console
. ~/.bashrc
```

### Check installation

```sh
nvcc --version
```

### Install COLMAP

For Linux:
```sh
# For Ubuntu 22.04
sudo apt-get install gcc-10 g++-10
export CC=/usr/bin/gcc-10
export CXX=/usr/bin/g++-10
export CUDAHOSTCXX=/usr/bin/g++-10

sudo apt-get install \
    git \
    cmake \
    ninja-build \
    build-essential \
    libboost-program-options-dev \
    libboost-filesystem-dev \
    libboost-graph-dev \
    libboost-system-dev \
    libeigen3-dev \
    libflann-dev \
    libfreeimage-dev \
    libmetis-dev \
    libgoogle-glog-dev \
    libgtest-dev \
    libsqlite3-dev \
    libglew-dev \
    qtbase5-dev \
    libqt5opengl5-dev \
    libcgal-dev \
    libceres-dev

git clone https://github.com/colmap/colmap.git
cd colmap
mkdir build
cd build
cmake .. -GNinja
ninja
sudo ninja install
```

## Installation

### Requirements

The software requirements are the following:
- Conda (recommended for easy setup)
- C++ Compiler for PyTorch extensions
- CUDA toolkit 11.8 for PyTorch extensions
- C++ Compiler and CUDA SDK must be compatible
- COLMAP
- FFMPEG

### Clone repository

```sh
git clone https://github.com/Anttwo/SuGaR.git --recursive
```

### Install the required Python packages

```sh
cd SuGaR
conda env create -f environment.yml
conda activate sugar
```

### Install the Gaussian Splatting rasterizer

```sh
cd gaussian_splatting/submodules/diff-gaussian-rasterization/
pip install -e .
cd ../simple-knn/
pip install -e .
cd ../../../
```

### Fixing bugs

Open file `sugar_scene/cameras.py`. In `line 31` the original code is
```python
with open(gs_output_path + 'cameras.json') as f:
```
Change it for:
```python
with open(gs_output_path + '/cameras.json') as f:
```

## Using Sugar

### Using COLMAP to preprocess dataset

Create a folder `location` and another inside of it called `input`. Put the input video inside of it.
```sh
# In case of the folders don't exist
mkdir location
cd location
mkdir input

# Extracting frames of input video
ffmpeg -i location/input/video.mp4 -qscale:v 1 -qmin 1 -vf fps=5 %04d.jpg
rm location/input/video.mp4
```

It is not mandatory to use a .mp4 video. It is possible to use with .mov and other formats.

In case of images with high resolutions, use the following code, only once, to resample images. 

This will prevent of using too much GPU memory.

In [ ]:
import os
from PIL import Image

path = "location/input"

# How much images will decrease
factor = 4

images_paths = [file for file in os.listdir(path) if file.endswith(".jpg")]
images_paths = sorted(images_paths)
for k, image_path in enumerate(images_paths):
    with Image.open(image_path) as im:
        M, N = im.size
        im.resize((M // factor, N // factor), Image.Resampling.LANCZOS).save(path + '/' + image_path[:-4] + '.jpg')

Then, use run COLMAP using the following code:

```sh
python3 gaussian_splatting/convert.py -s location
```

Depending on size of images, there is a chance of GPU memory not be enough. So it is necessary to resample image with code given in previous cell.

### Optimize a vanilla Gaussian Splatting model for 7k iterations

```sh
python3 gaussian_splatting/train.py -s location --iterations 7000 -m output
```

It is not necessary to create `output` folder.

### Optimize a SuGaR model

```sh
python3 train.py -s location -c output -r "density"
```

### Options for `train.py`

**Data and initial 3D Gaussian Splatting optimization**
| Parameter | Type | Description |
| :-------: | :--: | :---------: |
| `--scene_path` / `-s`   | `str` | Path to the source directory containing a COLMAP data set.|
| `--checkpoint_path` / `-c` | `str` | Path to the checkpoint directory of the vanilla 3D Gaussian Splatting model. |
| `--iteration_to_load` / `-i` | `int` | Iteration to load from the 3DGS checkpoint directory. If not specified, loads the iteration `7000`. |
| `--eval` | `bool` | If True, performs an evaluation split of the training images. Default is `True`. |

**SuGaR optimization**
| Parameter | Type | Description |
| :-------: | :--: | :---------: |
| `--regularization_type` / `-r` | `str` | Type of regularization to use for optimizing SuGaR. Can be `"density"` or `"sdf"`. |
| `--gpu` | `int` | Index of GPU device to use. Default is `0`. |

**Mesh extraction**
| Parameter | Type | Description |
| :-------: | :--: | :---------: |
| `--surface_level` / `-l` |`int`| Surface level to extract the mesh at. Default is `0.3`. |
| `--n_vertices_in_mesh` / `-v` | `int` | Number of vertices in the extracted mesh. Default is `1_000_000`. |
| `--bboxmin` / `-b` | `str` | Min coordinates to use for foreground bounding box, formatted as a string `"(x,y,z)"`.|
| `--bboxmax` / `-B` | `str` | Max coordinates to use for foreground bounding box, formatted as a string `"(x,y,z)"`. |
| `--center_bbox` | `bool` | If True, centers the bbox. Default is True. |

**SuGaR and mesh refinement (Hybrid representation)**
| Parameter | Type | Description |
| :-------: | :--: | :---------: |
| `--gaussians_per_triangle` / `-g` | `int` | Number of gaussians per triangle. Default is `1`. |
| `--refinement_iterations` / `-f` | `int` | Number of refinement iterations. Default is `15_000`. |    

**(Optional) Parameters for traditional textured mesh extraction**
| Parameter | Type | Description |
| :-------: | :--: | :---------: |
| `--export_uv_textured_mesh` / `-t` | `bool` | If True, will optimize and export a textured mesh as an .obj file from the refined SuGaR model. Computing a traditional colored UV texture should take less than 10 minutes. Default is `True`. |
| `--square_size` | `int` | Size of the square to use for the UV texture. Default is `10`. |
| `--postprocess_mesh` | `bool` | If True, postprocess the mesh by removing border triangles with low-density. This step takes a few minutes and is not needed in general, as it can also be risky. However, it increases the quality of the mesh in some cases, especially when very thin objects are visible only from one side in the images. Default is `False`. |
| `--postprocess_density_threshold` | `float` | Threshold to use for postprocessing the mesh. Default is `0.1`. |
| `--postprocess_iterations` | `int` | Number of iterations to use for postprocessing the mesh. Default is `5`. |

**(Optional) Parameters for exporting PLY files for the dedicated viewer**
| Parameter | Type | Description |
| :-------: | :--: | :---------: |
| `--export_ply` | `bool` | If True, export a `.ply` file with the refined 3D Gaussians at the end of the training. This file can be large (+/- 500MB), but is needed for using the dedicated viewer. Default is `True`. |

**(Optional) Default configurations**
| Parameter | Type | Description |
| :-------: | :--: | :---------: |
| `--low_poly` | `bool` | If True, uses standard config for a low poly mesh, with `200_000` vertices and `6` Gaussians per triangle. |
| `--high_poly` | `bool` | If True, uses standard config for a high poly mesh, with `1_000_000` vertices and `1` Gaussians per triangle. |
| `--refinement_time` | `str` | Default configs for time to spend on refinement. Can be `"short"` (2k iterations), `"medium"` (7k iterations) or `"long"` (15k iterations). |